In [15]:
!git clone https://www.github.com/yousefkoriem/NN-Project1.git

Cloning into 'NN-Project1'...
remote: Enumerating objects: 235, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 235 (delta 4), reused 29 (delta 2), pack-reused 202 (from 1)
Receiving objects: 100% (235/235), 190.33 MiB | 30.24 MiB/s, done.
Resolving deltas: 100% (58/58), done.


In [18]:
import os
os.chdir("/content/NN-Project1")

In [19]:
import keras
import tensorflow as tf
import numpy as np
import pandas as pd
from keras import layers
from keras.utils import text_dataset_from_directory
import spacy
import re
import pickle
from keras import layers, models, metrics, optimizers,callbacks

In [20]:
model = models.load_model("models/architecture/untrained_imdb_model.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'adamw', because it has 20 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [21]:
checkpoint = callbacks.ModelCheckpoint(
    filepath="models/architecture/best_imdb_model.keras",
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

In [22]:
early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    restore_best_weights=True,
    verbose=1
)

In [23]:
reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

In [24]:
# 1. Load the raw datasets
train_ds = tf.data.Dataset.load("data/processed/train_ds")
val_ds = tf.data.Dataset.load("data/processed/val_ds")
test_ds = tf.data.Dataset.load("data/processed/test_ds")

# 2. Define the label shape fix
def vectorize_text(text, label):
    return text, tf.expand_dims(label, -1)

# 3. Map the fix to the data
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
val_ds = val_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)
test_ds = test_ds.map(vectorize_text, num_parallel_calls=AUTOTUNE)

# 4. Apply cache and prefetch exactly ONCE at the very end
train_ds = train_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [25]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[checkpoint, early_stop, reduce_lr]
)

Epoch 1/100
622/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.6330 - f1_score: 0.6787 - loss: 0.5800
Epoch 1: val_loss improved from None to 0.30737, saving model to models/architecture/best_imdb_model.keras

Epoch 1: finished saving model to models/architecture/best_imdb_model.keras
625/625 ━━━━━━━━━━━━━━━━━━━━ 13s 12ms/step - accuracy: 0.7618 - f1_score: 0.7729 - loss: 0.4484 - val_accuracy: 0.8672 - val_f1_score: 0.8646 - val_loss: 0.3074 - learning_rate: 0.0010
Epoch 2/100
622/625 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8970 - f1_score: 0.8976 - loss: 0.2492
Epoch 2: val_loss did not improve from 0.30737
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.9279 - f1_score: 0.9277 - loss: 0.1855 - val_accuracy: 0.8692 - val_f1_score: 0.8758 - val_loss: 0.3778 - learning_rate: 0.0010
Epoch 3/100
622/625 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.9579 - f1_score: 0.9581 - loss: 0.1175
Epoch 3: val_loss did not improve from 0.30737
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 11

In [27]:
!mkdir results/tables

In [28]:
history_df = pd.DataFrame(history.history)
history_df.to_csv("results/tables/history.csv", index=False)

In [30]:
score = model.evaluate(test_ds)
score_df = pd.DataFrame([score], columns=["loss", "accuracy", "f1_score"])

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.8568 - f1_score: 0.8491 - loss: 0.3266


In [32]:
!mkdir results/models

In [33]:
model.save("results/models/final_imdb_model.keras")